# NTL Two Stage 8B Fold0
Inference only. Reuses the fixed checkpoint with plain + 0.35 centered log probabilities. No new classifier fitting, no Sinkhorn, and no test-label access. Validates against saved local probabilities before writing submission.parquet.

In [ ]:
import hashlib, json, os, sys, zipfile
from pathlib import Path
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
inputs=Path('/kaggle/input')
markers=list(inputs.rglob('bundle_manifest.json'))
if markers:
    bundle=markers[0].parent
else:
    archives=list(inputs.rglob('two_stage_8b_bundle.zip'))
    assert len(archives)==1, 'Attach the private checkpoint dataset'
    bundle=Path('/kaggle/working/two_stage_bundle')
    bundle.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:
        assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
        z.extractall(bundle)
manifest=json.loads((bundle/'bundle_manifest.json').read_text())
for name,digest in manifest['sha256'].items():
    assert hashlib.sha256((bundle/name).read_bytes()).hexdigest()==digest, name
base=None
for path in inputs.rglob('config.json'):
    try: cfg=json.loads(path.read_text())
    except (ValueError,OSError): continue
    if cfg.get('model_type')=='qwen3' and cfg.get('hidden_size')==4096 and cfg.get('num_hidden_layers')==36:
        base=path.parent
        break
assert base is not None, 'Attach Qwen3-8B-Base'
tests=[p for p in inputs.rglob('test.csv') if p.parent.name=='name-that-llm']
assert len(tests)==1, 'Attach the competition'
sys.path.insert(0,str(bundle))
from two_stage_inference import run
run(bundle,base,tests[0])
